In [ ]:
%cd ..

# Kaggle Episode Meta Explorer

Interactive exploration of aggregated match data from the Kaggle Daily Episode Datasets.

**Prerequisites:**
1. `make kaggle-all` — downloads + converts + aggregates the latest episode data
2. Parquet tables in `data/matches/aggregated/`

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)

INPUT = Path("data/matches/aggregated")

print("Loading tables...")
results = pd.read_parquet(INPUT / "results.parquet")
frames = pd.read_parquet(INPUT / "frames.parquet")
turns = pd.read_parquet(INPUT / "turn_summary.parquet")

print(f"Results: {len(results)} games")
print(f"Frames: {len(frames)} decision points")
print(f"Turn summaries: {len(turns)} player-turns")

if 'source' in results.columns:
    src_counts = results['source'].value_counts()
    for src, cnt in src_counts.items():
        print(f"  Source '{src}': {cnt} games")

## 1. Meta Composition

Deck archetype distribution among all games.

In [ ]:
# Archetype frequency
arches = pd.concat([
    results['arch0'].value_counts().rename('count'),
    results.groupby('arch0')['winner'].apply(lambda x: (x == 0).sum()).rename('wins'),
], axis=1).fillna(0).astype(int)
arches['win_rate'] = (arches['wins'] / arches['count']).round(3)
arches['play_rate'] = (arches['count'] / arches['count'].sum()).round(3)
arches = arches.sort_values('count', ascending=False)

display(arches)

# Pie chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].pie(arches['count'], labels=arches.index, autopct='%1.1f%%')
axes[0].set_title('Play Rate')
axes[1].barh(arches.index, arches['win_rate'])
axes[1].set_xlabel('Win Rate')
axes[1].set_title('Win Rate by Archetype')
for i, v in enumerate(arches['win_rate']):
    axes[1].text(v + 0.01, i, f'{v:.1%}', va='center')
plt.tight_layout()
plt.show()

## 2. Card Frequency

Which cards are most common across all decks?

In [ ]:
from collections import Counter

copy_counter = Counter()
deck_counter = Counter()
total_decks = 0

for col in ('deck0', 'deck1'):
    for deck_list in results[col]:
        if not hasattr(deck_list, '__iter__'):
            continue
        total_decks += 1
        seen = set()
        for cid in deck_list:
            copy_counter[int(cid)] += 1
            if int(cid) not in seen:
                deck_counter[int(cid)] += 1
                seen.add(int(cid))

# Display top cards
card_rows = []
for cid, copies in copy_counter.most_common(40):
    card_rows.append({
        'card_id': cid,
        'total_copies': copies,
        'n_decks': deck_counter[cid],
        'pct_decks': round(deck_counter[cid] / total_decks * 100, 1),
    })

card_df = pd.DataFrame(card_rows)
display(card_df.head(25))

# Plot: top 20 cards by deck presence
top20 = card_df.head(20)
plt.figure(figsize=(14, 8))
bars = plt.barh(range(len(top20)), top20['pct_decks'].values)
plt.yticks(range(len(top20)), [f"#{c}" for c in top20['card_id'].values])
plt.xlabel('% of Decks')
plt.title('Top 20 Cards by Deck Presence')
plt.gca().invert_yaxis()
for i, (v, c) in enumerate(zip(top20['pct_decks'], top20['total_copies'])):
    plt.text(v + 0.5, i, f'{v}%  ({c} copies)', va='center')
plt.tight_layout()
plt.show()

## 3. Prize Race

When do players take their first prize? How does the prize race develop?

In [ ]:
# Track when prize counts change per match
prize_changes = frames[
    (frames['me_prizes_taken'] > 0)
].copy()

# First prize taken per match
first_prize = prize_changes.groupby('match_id').agg(
    first_prize_turn=('turn', 'first'),
).reset_index()

merged = first_prize.merge(
    results[['match_id', 'arch0', 'arch1', 'winner', 'source']],
    on='match_id', how='left'
)

if not merged.empty and 'arch0' in merged.columns:
    # First prize distribution by archetype
    plt.figure(figsize=(12, 5))
    for arch in merged['arch0'].unique():
        subset = merged[merged['arch0'] == arch]['first_prize_turn']
        if len(subset) > 5:
            sns.kdeplot(subset, label=f"{arch} (n={len(subset)})", bw_adjust=0.5)
    plt.xlabel('Turn of First Prize Taken')
    plt.ylabel('Density')
    plt.title('First Prize Timing by Archetype')
    plt.legend()
    plt.show()

## 4. Game Length Distribution

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(results['turns'], bins=50, alpha=0.7, edgecolor='black')
plt.xlabel('Turns')
plt.ylabel('Games')
plt.title(f'Game Length Distribution (median={results["turns"].median():.0f})')

plt.subplot(1, 2, 2)
turns_by_arch = results.groupby('arch0')['turns'].agg(['mean', 'median', 'count']).sort_values('mean')
plt.barh(turns_by_arch.index, turns_by_arch['mean'], xerr=turns_by_arch['mean'] * 0.1)
plt.xlabel('Avg Turns')
plt.title('Average Game Length by Archetype')
for i, v in enumerate(turns_by_arch['mean']):
    plt.text(v + 0.5, i, f'{v:.1f}', va='center')

plt.tight_layout()
plt.show()

## 5. Action Heatmap by Archetype

Compare how different archetypes play — attack frequency, retreat rate, evolution speed.

In [ ]:
# Build action profile per archetype
turns_with_arch = turns.merge(
    results[['match_id', 'arch0']].drop_duplicates(),
    on='match_id', how='left'
)

action_cols = ['attacked', 'retreated', 'evolved', 'attached_energy', 'prize_taken']
profiles = turns_with_arch.groupby('arch0')[action_cols].mean().round(3)

plt.figure(figsize=(10, 6))
sns.heatmap(profiles, annot=True, fmt='.2f', cmap='YlOrRd', cbar_kws={'label': 'Avg per turn'})
plt.title('Action Rates per Turn by Archetype')
plt.tight_layout()
plt.show()

## 6. First vs Second Advantage

Win rate delta by archetype when going first vs second.

In [ ]:
# Get first player per match
first_players = turns.groupby('match_id')['first_player'].first().reset_index()

# For each match, determine if P0 won
matchup = results[['match_id', 'arch0', 'arch1', 'winner', 'source']].merge(
    first_players, on='match_id', how='left'
)

rows = []
for side, arch_col in [(0, 'arch0'), (1, 'arch1')]:
    for arch_name, group in matchup.groupby(arch_col):
        if pd.isna(arch_name) or not arch_name:
            continue
        first = group[group['first_player'] == side]
        second = group[group['first_player'] != side]
        if len(first) < 5 or len(second) < 5:
            continue
        rows.append({
            'archetype': arch_name,
            'first_wr': (first['winner'] == side).mean(),
            'second_wr': (second['winner'] == side).mean(),
            'delta': (first['winner'] == side).mean() - (second['winner'] == side).mean(),
            'n_first': len(first),
            'n_second': len(second),
        })

adv_df = pd.DataFrame(rows).sort_values('delta', ascending=False)

if not adv_df.empty:
    display(adv_df)
    
    plt.figure(figsize=(10, 4))
    x = range(len(adv_df))
    plt.bar(x, adv_df['delta'])
    plt.xticks(x, adv_df['archetype'], rotation=45, ha='right')
    plt.axhline(y=0, color='black', linestyle='--', linewidth=0.5)
    plt.ylabel('First Advantage (WR_delta)')
    plt.title('First Turn Advantage by Archetype')
    for i, (_, row) in enumerate(adv_df.iterrows()):
        plt.text(i, row['delta'] + 0.01 if row['delta'] >= 0 else row['delta'] - 0.03,
                 f"{row['delta']:+.1%}", ha='center', fontsize=9)
    plt.tight_layout()
    plt.show()

## 7. Evolution Timing

At what turn do players typically evolve their Pokemon?

In [ ]:
# Evolution events from frames
evolves = frames[frames['chosen_type'] == 9].copy()

if not evolves.empty:
    evolve_timing = evolves.merge(
        results[['match_id', 'arch0']], on='match_id', how='left'
    )
    
    plt.figure(figsize=(12, 4))
    for arch in evolve_timing['arch0'].unique():
        subset = evolve_timing[evolve_timing['arch0'] == arch]['turn']
        if len(subset) > 10:
            sns.kdeplot(subset, label=f"{arch} (n={len(subset)})", bw_adjust=0.5)
    plt.xlabel('Turn of Evolution')
    plt.ylabel('Density')
    plt.title('Evolution Timing by Archetype')
    plt.legend()
    plt.show()
    
    # Summary table
    evolve_summary = evolve_timing.groupby('arch0')['turn'].agg(['mean', 'median', 'count']).round(1)
    display(evolve_summary.sort_values('mean'))

## 8. Compare Our Agent vs Top Players

Side-by-side: how does exp005 (our best agent) compare with top Kaggle players?

In [ ]:
if 'source' in results.columns:
    kaggle = results[results['source'] == 'kaggle']
    local = results[results['source'] == 'local']
    
    # Win rates
    print("=== Win Rate Comparison ===")
    print(f"Kaggle top players (overall): {kaggle['winner'].value_counts().to_dict()}")
    print(f"Local (our agents): {local['winner'].value_counts().to_dict()}")
    
    # Game length
    print(f"\n=== Game Length ===")
    print(f"Kaggle: {kaggle['turns'].describe()}")
    print(f"\nLocal: {local['turns'].describe()}")
    
    # Archetype comparison
    kaggle_arches = pd.concat([kaggle['arch0'], kaggle['arch1']]).value_counts()
    local_arches = pd.concat([local['arch0'], local['arch1']]).value_counts()
    
    comp = pd.DataFrame({
        'kaggle_pct': (kaggle_arches / kaggle_arches.sum() * 100).round(1),
        'local_pct': (local_arches / local_arches.sum() * 100).round(1),
    }).fillna(0).sort_values('kaggle_pct', ascending=False)
    
    print("\n=== Archetype Comparison (Kaggle vs Local) ===")
    display(comp)
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(comp))
    w = 0.35
    ax.bar([i - w/2 for i in x], comp['kaggle_pct'], w, label='Kaggle', alpha=0.8)
    ax.bar([i + w/2 for i in x], comp['local_pct'], w, label='Local', alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(comp.index, rotation=45, ha='right')
    ax.set_ylabel('% of Games')
    ax.set_title('Archetype Distribution: Kaggle Top Players vs Our Agents')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 9. Deck Composition Analysis

What does a typical winning deck look like?

In [ ]:
# Compare winners vs losers: deck composition
winners = results[
    ((results['winner'] == 0) & (results['arch0'] != 'unknown')) |
    ((results['winner'] == 1) & (results['arch1'] != 'unknown'))
]
losers = results[
    ((results['winner'] != 0) & (results['arch0'] != 'unknown')) |
    ((results['winner'] != 1) & (results['arch1'] != 'unknown'))
]

print(f"Winners: {len(winners)}, Losers: {len(losers)}")

# Unique card counts as a rough proxy for deck diversity
def unique_count(deck_list):
    if hasattr(deck_list, '__iter__'):
        return len(set(deck_list))
    return 0

if len(winners) > 0 and len(losers) > 0:
    win_unique = pd.concat([
        winners[winners['winner'] == 0]['deck0'].apply(unique_count),
        winners[winners['winner'] == 1]['deck1'].apply(unique_count),
    ])
    lose_unique = pd.concat([
        losers[losers['winner'] != 0]['deck0'].apply(unique_count),
        losers[losers['winner'] != 1]['deck1'].apply(unique_count),
    ])
    
    print(f"\nAvg unique cards in winning decks: {win_unique.mean():.1f}")
    print(f"Avg unique cards in losing decks: {lose_unique.mean():.1f}")
    
    # Game length by winner
    print(f"\n=== Game Length by Winner ===")
    winner_games = results[results['winner'] >= 0]
    wl = winner_games.groupby('winner')['turns'].describe()
    display(wl)

## Key Insights

Summarize findings from the data above.

In [ ]:
print("=== Key Meta Insights ===")
print()

# 1. Most successful archetype
top_arch = arches[arches['win_rate'] == arches['win_rate'].max()].iloc[0]
print(f"1. Best performing archetype: {top_arch.name} ({top_arch['win_rate']:.1%} WR, {top_arch['count']} games)")

# 2. Most played archetype
pop_arch = arches.iloc[0]
print(f"2. Most popular archetype: {pop_arch.name} ({pop_arch['play_rate']:.1%} play rate)")

# 3. First turn advantage
if not adv_df.empty:
    best_first = adv_df.iloc[0]
    worst_first = adv_df.iloc[-1]
    print(f"3. Best first-turn advantage: {best_first['archetype']} ({best_first['delta']:+.1%})")
    print(f"   Worst first-turn advantage: {worst_first['archetype']} ({worst_first['delta']:+.1%})")

# 4. Average game length
print(f"4. Average game length: {results['turns'].mean():.1f} turns (median: {results['turns'].median():.0f})")

# 5. Fastest archetype
if not turns_by_arch.empty:
    fastest = turns_by_arch['mean'].idxmin()
    slowest = turns_by_arch['mean'].idxmax()
    print(f"5. Fastest games: {fastest} ({turns_by_arch.loc[fastest, 'mean']:.1f} avg turns)")
    print(f"   Slowest games: {slowest} ({turns_by_arch.loc[slowest, 'mean']:.1f} avg turns)")

# 6. Top universal cards
if not card_df.empty:
    universal = card_df[card_df['pct_decks'] > 80].head(10)
    if not universal.empty:
        print(f"6. Universal cards (>80% decks):")
        for _, row in universal.iterrows():
            print(f"   - Card #{row['card_id']} ({row['pct_decks']}% of decks)")

# 7. Action rate comparison
if not profiles.empty:
    print(f"7. Action rates per turn:")
    display(profiles)